# Optimización del Planograma de un Stand de Ventas — OXXO
## Modelo TA-BILP: *Template-Anchored Binary Integer Linear Program*

| Campo | Detalle |
|---|---|
| **Materia** | MA2008B — Optimización (Gpo. 603) |
| **Autores** | Mariel Álvarez · Viviana Carrizales · Jorge Andujo · Ana Ibarra · Álvaro Bolaños |
| **Profesores** | Fernando Elizalde · Monica Elizondo · Salvador García · Sofía Salinas |

---

### Descripción del Problema

Un **planograma** especifica qué productos se colocan en qué anaquel dentro de un mueble refrigerado de una tienda de conveniencia (OXXO). El problema consiste en:

> *Dado el surtido $P_t$ de una tienda, el tamaño del mueble $\tau$ (en puertas) y el tipo de mobiliario $\mu \in \{\text{CF}, \text{CFC}\}$, encontrar una asignación de productos a anaqueles que maximice la cobertura del surtido y la adherencia a reglas históricas de nivel, sujeta a restricciones físicas de capacidad.*

### Estructura del Notebook

| Sección | Contenido |
|---|---|
| 2 | Modelo matemático completo (TA-BILP) |
| 3 | Importación de librerías |
| 4 | Lectura y preparación de datos |
| 5 | Construcción del modelo |
| 6 | Resolución — ejemplos visuales |
| 7 | Tamaño del problema |
| 8 | Resultados estadísticos |
| 9 | Análisis de sensibilidad |
| 10 | Valor esperado de la ubicación del producto |
| 11 | Visualizaciones |
| 12 | Supuestos y conclusiones |

## 2. Modelo Matemático

### 2.1 Pipeline del Modelo TA-BILP

```
ENTRADAS                              MODELO                                    SALIDA
─────────                             ──────                                    ──────
H (planogramas históricos)  ──►  Paso 1: Minería de reglas  π_{pℓ}
Catálogo (w_p, h_p)        ──►  Paso 2: Selección template T* (P1)   ──►  Planograma
P_t (surtido objetivo)     ──►  Paso 3: Partición P_keep / P_new          generado
τ, μ, σ                   ──►  Paso 4: BILP → asignación óptima (P2)
```

---

### 2.2 Conjuntos

| Símbolo | Definición |
|---|---|
| $P$ | Catálogo completo de productos ($\lvert P \rvert = 470$) |
| $P_t \subseteq P$ | Productos disponibles en la tienda objetivo |
| $\mathcal{H}$ | Base de planogramas históricos ($\lvert \mathcal{H} \rvert = 5{,}614$) |
| $\mathcal{H}(\tau, \mu)$ | Subconjunto con tamaño $\tau$ y tipo de mueble $\mu$ |
| $\mathcal{H}(\mu, \sigma)$ | Subconjunto con tipo $\mu$ y segmento $\sigma$ |
| $S$ | Charolas del template seleccionado ($\lvert S \rvert \in [5, 38]$) |
| $L = \{0,\ldots,6\}$ | Niveles verticales (0 = piso, 6 = techo) |
| $P_T$ | Productos del template $T^*$ |
| $P_{\text{keep}} = P_t \cap P_T$ | Productos conservados en su posición original |
| $P_{\text{gone}} = P_T \setminus P_t$ | Productos removidos (liberan espacio en sus charolas) |
| $P_{\text{new}} = P_t \setminus P_T$ | Productos nuevos a ubicar mediante el BILP |

---

### 2.3 Parámetros

| Símbolo | Descripción |
|---|---|
| $w_p$ | Ancho del producto $p$ (cm) |
| $h_p$ | Altura del producto $p$ (cm) |
| $f_p$ | Número de frentes de $p$ en el template (heredado) |
| $W_s$ | Ancho total del anaquel $s$ (cm) |
| $H_s$ | Altura efectiva del anaquel $s$ (gap entre coordenadas $Y$ consecutivas, cm) |
| $\pi_{p\ell}$ | Probabilidad histórica de que el producto $p$ esté en el nivel $\ell$ |
| $\lambda_1,\, \lambda_2 \geq 0$ | Pesos de adherencia de nivel y afinidad dimensional (default = 1.0) |

#### Paso 1 — Minería de reglas $\pi_{p\ell}$

$$\pi_{p\ell}^{(\mu,\sigma)} = \frac{\text{# veces que } p \text{ aparece en nivel } \ell \text{ en } \mathcal{H}(\mu,\sigma)}{\text{total de apariciones de } p \text{ en } \mathcal{H}(\mu,\sigma)}$$

Si $\lvert\mathcal{H}(\mu,\sigma)\rvert < 20$, se usa $\mathcal{H}(\mu)$ como fallback para evitar reglas poco confiables.

#### Paso 3 — Ancho residual por anaquel

$$W_s^{\text{rem}} = W_s - \sum_{\substack{p \in P_{\text{keep}} \\ s_p^T = s}} w_p \cdot f_p \quad \forall s \in S$$

#### Paso 4 — Función de afinidad $\gamma(p, s)$

**Distancia dimensional** entre dos productos:
$$\delta(p, q) = |w_p - w_q| + 2\,|h_p - h_q|$$

**Si la charola $s$ tiene productos liberados** $P_{\text{gone}}(s) \neq \emptyset$ (afinidad de sustitución):
$$\gamma(p, s) = \max\!\left(0,\; 1 - \frac{\displaystyle\min_{q \in P_{\text{gone}}(s)} \delta(p,q)}{\delta_{\max}}\right)$$

**Si no hay productos liberados** (afinidad de posición):
$$\gamma(p, s) = \max\!\left(0,\; 1 - \frac{|y_s - y^*_{\text{ideal}}(h_p)|}{y_{\max} - y_{\min}}\right)$$

donde $y^*_{\text{ideal}}(h_p)$ se deriva de la correlación observada $r = -0.90$ entre altura del producto y coordenada $Y$ del anaquel.

---

### 2.4 Variables de Decisión

$$\boxed{x_{ps} \in \{0, 1\} \quad \forall p \in P_{\text{new}},\; s \in S}$$

$x_{ps} = 1$ indica que el producto $p$ se asigna al anaquel $s$. Los productos de $P_{\text{keep}}$ están **fijos** en sus posiciones del template (no se optimizan).

---

### 2.5 Función Objetivo — Problema P2

$$\max_{x}\;\;
\underbrace{\sum_{p \in P_{\text{new}}} \sum_{s \in S} x_{ps}}_{\Phi_{\text{cob}}}
\;+\; \lambda_1 \cdot \underbrace{\frac{1}{|P_{\text{new}}|}\sum_{p} \sum_{s} \pi_{p,\ell_s} \cdot x_{ps}}_{\Phi_{\text{nivel}}}
\;+\; \lambda_2 \cdot \underbrace{\frac{1}{|P_{\text{new}}|}\sum_{p} \sum_{s} \gamma(p,s) \cdot x_{ps}}_{\Phi_{\text{afinidad}}}$$

| Componente | Interpretación |
|---|---|
| $\Phi_{\text{cob}}$ | Cobertura bruta: número de productos nuevos efectivamente colocados |
| $\Phi_{\text{nivel}}$ | Adherencia histórica: probabilidad promedio de que cada producto esté en su nivel preferido |
| $\Phi_{\text{afinidad}}$ | Ajuste dimensional: qué tan bien encaja cada producto en la charola asignada |

---

### 2.6 Restricciones

**(C1) Unicidad de asignación** — cada producto nuevo va en máximo un anaquel:
$$\sum_{s \in S} x_{ps} \leq 1 \quad \forall p \in P_{\text{new}}$$

**(C2) Capacidad de ancho residual** — el espacio ocupado no excede el disponible:
$$\sum_{p \in P_{\text{new}}} w_p \cdot x_{ps} \leq W_s^{\text{rem}} \quad \forall s \in S$$

**(C3) Compatibilidad de altura** — el producto no puede superar la altura del anaquel (tolerancia 5%):
$$x_{ps} = 0 \quad \forall (p, s): h_p > 1.05 \cdot H_s$$

---

### 2.7 Selección de Template — Problema P1

$$T^* = \arg\max_{T \in \mathcal{H}(\tau, \mu)} \lvert P_T \cap P_t \rvert$$

Enumeración finita con complejidad $O(\lvert\mathcal{H}\rvert \cdot \lvert P_t\rvert)$. Garantiza que el template inicial sea el más similar al surtido objetivo.

---

### 2.8 Complejidad Típica

| Medida | Valor (20% swap, $\tau = 4.0$) |
|---|---|
| Variables $\lvert x \rvert$ | $\lvert P_{\text{new}}\rvert \times \lvert S\rvert \approx 26 \times 24 = 624$ |
| Restricciones | $\lvert P_{\text{new}}\rvert + \lvert S\rvert + \text{altura} \approx 210$ |
| Tiempo de solución | $< 0.2$ s (solver CBC, gap $\leq 1\%$) |

## 3. Importación de Librerías

In [1]:
# Librerías estándar y científicas
import os, sys, time, random, warnings, datetime
import platform
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict

# Suprimir warnings de PuLP y numpy
warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# Asegurar que el directorio del notebook esté en sys.path
# para encontrar los módulos locales (exploracion, planogram_model)
_nb_dir = os.path.abspath('.')
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

# Módulos locales del proyecto
from exploracion import load_planograms, Planogram, Shelf, ProductPlacement
from planogram_model import (
    build_product_catalog,
    mine_placement_rules,
    evaluate_rule_adherence,
    ProductInfo,
)
import pulp

# Directorio de salida para imágenes y CSV
OUTPUT_DIR = "test_output/final_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Función auxiliar de logging con timestamp
def log(msg):
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {msg}")

print(f"Python  : {sys.version.split()[0]}")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"PuLP    : {pulp.__version__}")
print(f"Hardware: {platform.processor()}")
print(f"OS      : {platform.system()} {platform.release()}")

Python  : 3.14.4
NumPy   : 2.4.4
Pandas  : 3.0.2
PuLP    : 3.3.0
Hardware: arm
OS      : Darwin 24.6.0


## 4. Lectura y Preparación de Datos

### Fuente de Datos

Los datos provienen de dos archivos CSV que contienen planogramas históricos de tiendas OXXO:

| Archivo | Descripción |
|---|---|
| `datos/ejemplo_planograma.csv` | Planogramas del conjunto principal (43 planogramas de referencia) |
| `datos/Ejemplo 2.csv` | Planogramas adicionales que amplían el histórico |

Cada fila en los CSV representa la colocación de un producto en un anaquel: incluye el UPC, descripción, dimensiones ($w_p$, $h_p$), posición ($x$, $y$), charola, nivel, número de frentes y metadatos del mueble (tipo, tamaño, segmento).

### Procesamiento

1. Se cargan ambos archivos con `load_planograms()`, que construye objetos `Planogram → Shelf → ProductPlacement`.
2. Se construye el catálogo con `build_product_catalog()`, que consolida las dimensiones únicas de cada UPC.
3. Se unen los dos conjuntos para formar la base histórica $\mathcal{H}$ completa.

In [2]:
log("Cargando datos...")
t0_load = time.perf_counter()

# Planogramas de referencia (43) y planogramas adicionales
planograms_orig = load_planograms("datos/ejemplo_planograma.csv")
planograms_new  = load_planograms("datos/Ejemplo 2.csv")

# Base histórica completa H
planograms = planograms_orig + planograms_new

# Catálogo de productos: {UPC: ProductInfo(width, height, description)}
catalog = build_product_catalog(["datos/ejemplo_planograma.csv",
                                  "datos/Ejemplo 2.csv"])

# Resumen de dimensiones
muebles   = sorted({p.mueble_id   for p in planograms})
segmentos = sorted({p.segmento_id for p in planograms})
tamanos   = sorted({p.tamano      for p in planograms})

print(f"\nDIMENSIONES DEL PROBLEMA")
print(f"{'='*55}")
print(f"  |H|             = {len(planograms)}")
print(f"  |Catálogo| (|P|)= {len(catalog)}")
print(f"  Muebles (μ)     = {muebles}")
print(f"  Segmentos (σ)   = {segmentos}")
print(f"  Tamaños (τ)     = {tamanos}")
print()
for m in muebles:
    mp = [p for p in planograms if p.mueble_id == m]
    print(f"  {m}: {len(mp):>5} planogramas")

log(f"  Carga completada en {time.perf_counter()-t0_load:.1f}s")

# Pool total de UPCs disponibles (para simulaciones)
all_pool = {p.upc
            for pl in planograms
            for s in pl.shelves.values()
            for p in s.products}
print(f"\n  UPCs únicos en el pool histórico: {len(all_pool)}")

[18:19:59] Cargando datos...

NORMALIZED COLUMNS:
  SEGMENTO_ID
  MUEBLE_ID
  PLANOGRUPO
  PLAN_EJECUCION
  CONJUNTO_ID
  TAMANO_POST
  DIRECCION_LEGO_ID
  UPC_CVE
  ITEM_DESC
  CHAROLA
  UBICACION_BANDEJA
  NUM_FRENTES
  ALTO
  ANCHO
  WIDTH
  HEIGHT
  X
  Y
  DISENO_REFERENCIA
  SEPARADOR
  SQUEEZE
  STAGE_ID
  EXECUTION_DATE
  TYPE
  DESC1
  UNIDADES_X_FRENTE
  PROFUNDO
  CATEGORIA_PRODUCTO
  LEGO_ID
  CAMAS

Loaded 43 planograms.

NORMALIZED COLUMNS:
  SEGMENTO_ID
  MUEBLE_ID
  PLANOGRUPO
  TAMANO_POST
  DIRECCION_LEGO_ID
  UPC_CVE
  NUM_FRENTES
  CHAROLA
  UBICACION_BANDEJA
  ANCHO
  ITEM_DESC
  ALTO
  PROFUNDO

Spatial columns (X, Y, WIDTH, HEIGHT) not found — deriving layout from CHAROLA + TAMANO

Variant reconstruction: 5893 total → 5571 kept (>= 95% coverage)

Loaded 5571 planograms.

DIMENSIONES DEL PROBLEMA
  |H|             = 5614
  |Catálogo| (|P|)= 470
  Muebles (μ)     = ['CF', 'CFC']
  Segmentos (σ)   = ['BCO', 'CLA', 'HOG', 'HRN', 'OFC', 'PTC', 'REC', 'RET', 'SIN']
  T

## 5. Construcción del Modelo

El modelo TA-BILP se implementa en dos funciones principales:
- `ideal_y(height)` — calcula la coordenada $Y$ ideal para un producto dado su altura.
- `generate_planogram(...)` — función principal que encapsula los 4 pasos del modelo.
- `_build_result(...)` — construye el objeto `Planogram` final a partir de la solución.

### 5.1 Función Auxiliar: Posición Ideal por Altura

Deriva la coordenada $Y$ ideal de un producto basándose en la correlación empírica $r = -0.90$ entre altura y posición vertical (productos más altos van en posiciones más bajas del mueble).

In [3]:
def ideal_y(height: float) -> float:
    """Coordenada Y ideal según la altura del producto.

    Derivado de la correlación Pearson r=-0.90 (alto vs Y):
    productos más altos tienden a estar en anaqueles más bajos (Y pequeño).
    """
    if   height >= 30: return  20.0
    elif height >= 25: return  90.0
    elif height >= 20: return 130.0
    elif height >= 15: return 166.0
    else:              return 168.0

### 5.2 Paso 1 — Minería de Reglas $\pi_{p\ell}$

La función `mine_placement_rules()` calcula las probabilidades de nivel para cada producto a partir de los planogramas históricos del segmento/mueble correspondiente.

### 5.3 Paso 2 — Selección del Template (P1)

Enumeración finita sobre $\mathcal{H}(\tau, \mu)$ para encontrar el planograma con mayor overlap con $P_t$.

### 5.4 Paso 3 — Partición y Preprocesamiento

Cálculo de $P_{\text{keep}}$, $P_{\text{gone}}$, $P_{\text{new}}$ y del ancho residual $W_s^{\text{rem}}$ por anaquel.

### 5.5 Paso 4 — BILP (P2)

Construcción y resolución del problema de programación entera binaria con PuLP/CBC.

In [4]:
def generate_planogram(
    target_upcs: set,
    target_tamano: float,
    target_mueble: str,
    target_segmento: str,
    historic_planograms: list,
    catalog: dict,
    lambda1: float = 1.0,
    lambda2: float = 1.0,
    min_planograms_for_rules: int = 20,
    time_limit: int = 300,
    gap_rel: float = 0.01,
) -> dict:
    """Modelo TA-BILP completo (auto-contenido).

    Parámetros:
        target_upcs         : surtido objetivo P_t
        target_tamano       : tamaño del mueble τ
        target_mueble       : tipo de mueble μ ('CF' o 'CFC')
        target_segmento     : segmento de tienda σ
        historic_planograms : base histórica H
        catalog             : catálogo {UPC: ProductInfo}
        lambda1, lambda2    : pesos de la función objetivo

    Retorna:
        dict con 'planogram', 'status', métricas y tiempos
    """
    t0 = time.perf_counter()

    # ── PASO 1: MINERÍA DE REGLAS π_{pℓ} ──────────────────────────────
    seg_plans = [p for p in historic_planograms
                 if p.mueble_id == target_mueble
                 and p.segmento_id == target_segmento]

    if len(seg_plans) >= min_planograms_for_rules:
        rules = mine_placement_rules(seg_plans, catalog)
        rules_src = f"{target_mueble}/{target_segmento} ({len(seg_plans)} planos)"
    else:
        # Fallback: usar todos los planogramas del mismo tipo de mueble
        mueble_plans = [p for p in historic_planograms
                        if p.mueble_id == target_mueble]
        rules = mine_placement_rules(mueble_plans, catalog)
        rules_src = f"{target_mueble} fallback ({len(mueble_plans)} planos)"

    lp = rules['level_probs']  # {upc: {level: prob}}

    # ── PASO 2: SELECCIÓN DE TEMPLATE T* (P1) ─────────────────────────
    best_tpl, best_ov, best_tot = None, -1, 0
    for plano in historic_planograms:
        if plano.tamano != target_tamano or plano.mueble_id != target_mueble:
            continue
        tpl_upcs = {p.upc for s in plano.shelves.values() for p in s.products}
        ov = len(tpl_upcs & target_upcs)
        if ov > best_ov:
            best_ov, best_tpl, best_tot = ov, plano, len(tpl_upcs)

    if best_tpl is None:
        return {'planogram': None, 'status': 'Infeasible',
                'time_total': time.perf_counter() - t0,
                'n_placed': 0, 'n_keep': 0, 'n_new': len(target_upcs)}

    template = best_tpl

    # ── PASO 3: PARTICIÓN ─────────────────────────────────────────────
    shelves = {}
    tpl_assign  = {}   # {upc: charola_id}
    tpl_facings = {}   # {upc: facings}

    for ch, shelf in template.shelves.items():
        shelves[ch] = shelf
        for p in shelf.products:
            tpl_assign[p.upc]  = ch
            tpl_facings[p.upc] = p.facings

    P_T     = set(tpl_assign.keys())
    P_keep  = target_upcs & P_T
    P_gone  = P_T - target_upcs
    P_new   = sorted([u for u in (target_upcs - P_T) if u in catalog])
    P_gone_s = sorted([u for u in P_gone if u in catalog])

    # Ancho residual W_s^rem para cada charola
    S_list = sorted(shelves.keys())
    rem_w  = {}
    for ch in S_list:
        kw = sum(catalog[p.upc].width * p.facings
                 for p in shelves[ch].products
                 if p.upc in P_keep and p.upc in catalog)
        rem_w[ch] = shelves[ch].shelf_width - kw

    # Altura efectiva H_s (gap entre Y consecutivos)
    ys    = sorted({s.y for s in shelves.values()})
    eff_h = {}
    for i, yv in enumerate(ys):
        eff_h[yv] = (ys[i+1] - yv) if i < len(ys)-1                     else (ys[i] - ys[i-1] if i > 0 else 40.0)

    n_new = len(P_new)

    # ── PASO 4: BILP (P2) ─────────────────────────────────────────────
    if n_new == 0:  # Caso trivial: todos los productos del surtido ya están
        result = _build_result(template, shelves, P_keep, {}, catalog, tpl_facings)
        return {'planogram': result, 'status': 'Optimal (trivial)',
                'time_total': time.perf_counter() - t0, 'time_bilp': 0,
                'n_placed': len(P_keep), 'n_keep': len(P_keep),
                'n_new': 0, 'n_new_placed': 0, 'n_gone': len(P_gone),
                'template': template.title, 'overlap': best_ov,
                'rules_src': rules_src, 'rules': rules,
                'n_vars': 0, 'n_cons': 0,
                'phi_nivel': 0, 'phi_aff': 0}

    # Calcular γ(p, s) — afinidad dimensional
    gone_shelf = defaultdict(list)
    for u in P_gone_s:
        if u in tpl_assign:
            gone_shelf[tpl_assign[u]].append(u)

    def delta(p, q):
        return (abs(catalog[p].width  - catalog[q].width) +
                2 * abs(catalog[p].height - catalog[q].height))

    d_all  = [delta(p, q) for p in P_new for q in P_gone_s]
    d_max  = max(d_all) if d_all else 1.0
    y_range = max(ys) - min(ys) if len(ys) > 1 else 1.0

    gamma = {}
    for p in P_new:
        for ch in S_list:
            gone = gone_shelf.get(ch, [])
            if gone:
                # Afinidad de sustitución dimensional
                gamma[p, ch] = max(0, 1 - min(delta(p, q) for q in gone) / d_max)
            else:
                # Afinidad de posición vertical
                gamma[p, ch] = max(0, 1 - abs(shelves[ch].y - ideal_y(catalog[p].height)) / y_range)

    # Construir el BILP con PuLP
    prob = pulp.LpProblem("TA_BILP", pulp.LpMaximize)

    # Variables binarias x_{ps}
    x = {(p, ch): pulp.LpVariable(f"x_{p}_{ch}", cat="Binary")
         for p in P_new for ch in S_list}

    # Componentes de la función objetivo
    phi_cob = pulp.lpSum(x[p, ch] for p in P_new for ch in S_list)

    phi_niv = (1 / n_new) * pulp.lpSum(
        lp.get(p, {}).get(shelves[ch].level, 0) * x[p, ch]
        for p in P_new for ch in S_list)

    phi_aff = (1 / n_new) * pulp.lpSum(
        gamma.get((p, ch), 0) * x[p, ch]
        for p in P_new for ch in S_list)

    # FO: maximizar cobertura + adherencia de nivel + afinidad
    prob += phi_cob + lambda1 * phi_niv + lambda2 * phi_aff

    # (C1) Cada producto en máximo un anaquel
    for p in P_new:
        prob += pulp.lpSum(x[p, ch] for ch in S_list) <= 1

    # (C2) Capacidad de ancho residual
    for ch in S_list:
        prob += pulp.lpSum(catalog[p].width * x[p, ch] for p in P_new) <= rem_w[ch]

    # (C3) Compatibilidad de altura (con tolerancia 5%)
    for p in P_new:
        for ch in S_list:
            if catalog[p].height > eff_h.get(shelves[ch].y, shelves[ch].shelf_height) * 1.05:
                prob += x[p, ch] == 0

    nv, nc = len(x), len(prob.constraints)

    # Resolver con CBC
    solver = pulp.PULP_CBC_CMD(timeLimit=time_limit, msg=0, gapRel=gap_rel)
    t_bilp = time.perf_counter()
    prob.solve(solver)
    t_bilp = time.perf_counter() - t_bilp

    # Extraer asignación óptima
    assign = {}
    for p in P_new:
        for ch in S_list:
            v = pulp.value(x[p, ch])
            if v and v > 0.5:
                assign[p] = ch
                break

    result = _build_result(template, shelves, P_keep, assign, catalog, tpl_facings)

    return {
        'planogram'   : result,
        'status'      : pulp.LpStatus[prob.status],
        'time_total'  : time.perf_counter() - t0,
        'time_bilp'   : t_bilp,
        'n_placed'    : len(P_keep) + len(assign),
        'n_keep'      : len(P_keep),
        'n_new'       : n_new,
        'n_new_placed': len(assign),
        'n_gone'      : len(P_gone),
        'template'    : template.title,
        'overlap'     : best_ov,
        'rules_src'   : rules_src,
        'rules'       : rules,
        'n_vars'      : nv,
        'n_cons'      : nc,
        'phi_nivel'   : pulp.value(phi_niv),
        'phi_aff'     : pulp.value(phi_aff),
    }

### 5.6 Construcción del Planograma Final

In [5]:
def _build_result(template, shelves, P_keep, assign, catalog, tpl_facings):
    """Construye el objeto Planogram final combinando P_keep y la asignación óptima."""
    r = Planogram(template.segmento_id, template.mueble_id,
                  template.planogrupo, template.tamano,
                  template.direccion, "TA-BILP")

    for ch, shelf in shelves.items():
        ns = Shelf(shelf.charola, shelf.door, shelf.level,
                   shelf.x, shelf.y, shelf.shelf_width, shelf.shelf_height)

        # Agregar productos conservados (P_keep) con sus facings originales
        for p in shelf.products:
            if p.upc in P_keep:
                ns.products.append(ProductPlacement(
                    p.upc, p.description, p.shelf, p.position,
                    p.facings, p.width, p.height))

        # Agregar productos nuevos asignados por el BILP (P_new → x_{ps}=1)
        for upc, ach in assign.items():
            if ach == ch and upc in catalog:
                ci  = catalog[upc]
                pos = max((p.position for p in ns.products), default=0) + 1
                ns.products.append(ProductPlacement(
                    upc, ci.description, ch, pos, 1, ci.width, ci.height))

        r.shelves[ch] = ns

    return r

## 6. Resolución del Modelo — Ejemplos Visuales

Se generan planogramas para dos tiendas con características distintas y se evalúan las métricas de adherencia al histórico.

### 6.1 Ejemplo 1: Tienda BCO, Mueble CF, $\tau = 4.0$, 20% de productos intercambiados

In [6]:
log("=" * 60)
log("EJEMPLO 1: BCO / CF / τ=4.0 / 20% swap")
log("=" * 60)

# Planograma base: primer planograma del conjunto original
ex1 = planograms_orig[0]
orig_upcs = {p.upc for s in ex1.shelves.values() for p in s.products}

# Simular un surtido diferente: reemplazar 20% de los productos
random.seed(42)
n_swap    = int(len(orig_upcs) * 0.20)
to_remove = set(random.sample(sorted(orig_upcs), n_swap))
replacements = set(random.sample(sorted(all_pool - orig_upcs), n_swap))
target1   = (orig_upcs - to_remove) | replacements

# Ejecutar el modelo
r1 = generate_planogram(
    target_upcs=target1,
    target_tamano=ex1.tamano,
    target_mueble=ex1.mueble_id,
    target_segmento=ex1.segmento_id,
    historic_planograms=planograms,
    catalog=catalog,
)

print(f"\n  INPUT   : |P_t|={len(target1)}, τ={ex1.tamano}, μ={ex1.mueble_id}, σ={ex1.segmento_id}")
print(f"  TEMPLATE: {r1['template']} (overlap={r1['overlap']})")
print(f"  PARTICIÓN: P_keep={r1['n_keep']}, P_gone={r1['n_gone']}, P_new={r1['n_new']}")
print(f"  OUTPUT  : {r1['n_placed']} colocados ({r1['n_new_placed']}/{r1['n_new']} nuevos)")
print(f"  STATUS  : {r1['status']}")
print(f"  Tiempo  : BILP={r1['time_bilp']:.3f}s  total={r1['time_total']:.3f}s")
print(f"  Problema: {r1['n_vars']} vars · {r1['n_cons']} restricciones")

metrics1 = evaluate_rule_adherence(r1['planogram'], r1['rules'], catalog)
print(f"\n  MÉTRICAS DE ADHERENCIA:")
for k, v in metrics1.items():
    print(f"    {k:<28} {v:.4f}")

# Guardar visualizaciones
ex1.plot_planogram(save_path=os.path.join(OUTPUT_DIR, "ej1_original.png"))
r1['planogram'].plot_planogram(save_path=os.path.join(OUTPUT_DIR, "ej1_generado.png"))
log(f"  Imágenes guardadas en {OUTPUT_DIR}/")

[18:20:09] ============================================================
[18:20:09] EJEMPLO 1: BCO / CF / τ=4.0 / 20% swap
[18:20:09] ============================================================

  INPUT   : |P_t|=131, τ=4.0, μ=CF, σ=BCO
  TEMPLATE: BCO | CF | Refrescos | Tamaño=4.0 | Dir=DI | Conjunto=10MON (overlap=105)
  PARTICIÓN: P_keep=105, P_gone=26, P_new=26
  OUTPUT  : 129 colocados (24/26 nuevos)
  STATUS  : Optimal
  Tiempo  : BILP=1.103s  total=1.159s
  Problema: 624 vars · 210 restricciones

  MÉTRICAS DE ADHERENCIA:
    level_rule_score             0.5036
    level_mode_accuracy          0.7480
    adjacency_hit_rate           0.7621
    adjacency_weighted           0.0525
    width_feasibility            1.0000
    height_level_corr            -0.8759
    n_products_placed            129.0000
Saved to test_output/final_model/ej1_original.png
Saved to test_output/final_model/ej1_generado.png
[18:20:11]   Imágenes guardadas en test_output/final_model/


### 6.2 Ejemplo 2: Tienda HRN, Mueble CF, $\tau = 5.0$, 30% de productos intercambiados

In [7]:
log("=" * 60)
log("EJEMPLO 2: HRN / CF / τ=5.0 / 30% swap")
log("=" * 60)

# Buscar planograma HRN con tamaño 5.0; si no existe, usar cualquier HRN
ex2 = next((p for p in planograms_orig
            if p.segmento_id == 'HRN' and p.tamano == 5.0), None)
if ex2 is None:
    ex2 = next(p for p in planograms_orig if p.segmento_id == 'HRN')

orig_upcs2 = {p.upc for s in ex2.shelves.values() for p in s.products}
random.seed(123)
n_swap2      = int(len(orig_upcs2) * 0.30)
to_remove2   = set(random.sample(sorted(orig_upcs2), n_swap2))
replacements2 = set(random.sample(sorted(all_pool - orig_upcs2),
                                   min(n_swap2, len(all_pool - orig_upcs2))))
target2 = (orig_upcs2 - to_remove2) | replacements2

r2 = generate_planogram(
    target_upcs=target2,
    target_tamano=ex2.tamano,
    target_mueble=ex2.mueble_id,
    target_segmento=ex2.segmento_id,
    historic_planograms=planograms,
    catalog=catalog,
)

print(f"\n  INPUT   : |P_t|={len(target2)}, τ={ex2.tamano}, μ={ex2.mueble_id}, σ={ex2.segmento_id}")
print(f"  TEMPLATE: {r2['template']} (overlap={r2['overlap']})")
print(f"  PARTICIÓN: P_keep={r2['n_keep']}, P_gone={r2['n_gone']}, P_new={r2['n_new']}")
print(f"  OUTPUT  : {r2['n_placed']} colocados ({r2['n_new_placed']}/{r2['n_new']} nuevos)")
print(f"  STATUS  : {r2['status']}, t_BILP={r2['time_bilp']:.3f}s")

metrics2 = evaluate_rule_adherence(r2['planogram'], r2['rules'], catalog)
print(f"\n  MÉTRICAS DE ADHERENCIA:")
for k, v in metrics2.items():
    print(f"    {k:<28} {v:.4f}")

ex2.plot_planogram(save_path=os.path.join(OUTPUT_DIR, "ej2_original.png"))
r2['planogram'].plot_planogram(save_path=os.path.join(OUTPUT_DIR, "ej2_generado.png"))
log(f"  Imágenes guardadas en {OUTPUT_DIR}/")

[18:20:11] ============================================================
[18:20:11] EJEMPLO 2: HRN / CF / τ=5.0 / 30% swap
[18:20:11] ============================================================

  INPUT   : |P_t|=151, τ=5.0, μ=CF, σ=HRN
  TEMPLATE: HRN | CF | Refrescos | Tamaño=5.0 | Dir=ID | Conjunto=10MON (overlap=106)
  PARTICIÓN: P_keep=106, P_gone=45, P_new=45
  OUTPUT  : 150 colocados (44/45 nuevos)
  STATUS  : Optimal, t_BILP=0.317s

  MÉTRICAS DE ADHERENCIA:
    level_rule_score             0.5023
    level_mode_accuracy          0.7324
    adjacency_hit_rate           0.6448
    adjacency_weighted           0.0310
    width_feasibility            1.0000
    height_level_corr            -0.8749
    n_products_placed            150.0000
Saved to test_output/final_model/ej2_original.png
Saved to test_output/final_model/ej2_generado.png
[18:20:13]   Imágenes guardadas en test_output/final_model/


## 7. Tamaño del Problema

Esta sección cuantifica las dimensiones del problema de optimización para el caso de uso típico y para el conjunto de evaluación completo.

La tabla incluye:
- **Nodos / Conjuntos**: cardinalidad de cada conjunto matemático.
- **Variables**: número de variables de decisión $x_{ps}$ del BILP.
- **Parámetros**: número de parámetros del modelo.
- **Restricciones**: número de restricciones activas.

In [8]:
# ── Tamaño del modelo en el Ejemplo 1 (caso representativo) ──────────
ex_upcs  = target1
ex_tamano = ex1.tamano
ex_mueble = ex1.mueble_id
ex_seg    = ex1.segmento_id

# Reconstruir partición del ejemplo 1 para calcular tamaños exactos
ex_template = None
best_ov_ex  = -1
for plano in planograms:
    if plano.tamano != ex_tamano or plano.mueble_id != ex_mueble:
        continue
    tpl_upcs = {p.upc for s in plano.shelves.values() for p in s.products}
    ov = len(tpl_upcs & ex_upcs)
    if ov > best_ov_ex:
        best_ov_ex, ex_template = ov, plano

ex_P_T    = {p.upc for s in ex_template.shelves.values() for p in s.products}
ex_P_keep = ex_upcs & ex_P_T
ex_P_gone = ex_P_T - ex_upcs
ex_P_new  = [u for u in (ex_upcs - ex_P_T) if u in catalog]
ex_S      = list(ex_template.shelves.keys())

# Número de restricciones de altura (C3) — pre-filtra variables
ex_ys    = sorted({s.y for s in ex_template.shelves.values()})
ex_eff_h = {}
for i, yv in enumerate(ex_ys):
    ex_eff_h[yv] = (ex_ys[i+1]-yv) if i < len(ex_ys)-1 else (ex_ys[i]-ex_ys[i-1] if i>0 else 40.0)

n_height_cons = sum(
    1 for p in ex_P_new for ch in ex_S
    if catalog[p].height > ex_eff_h.get(ex_template.shelves[ch].y, ex_template.shelves[ch].shelf_height) * 1.05
)

# Número de reglas π_{pℓ} (parámetros del modelo)
seg_plans_ex = [p for p in planograms if p.mueble_id == ex_mueble and p.segmento_id == ex_seg]
if len(seg_plans_ex) < 20:
    seg_plans_ex = [p for p in planograms if p.mueble_id == ex_mueble]
rules_ex  = mine_placement_rules(seg_plans_ex, catalog)
n_rules   = sum(len(v) for v in rules_ex['level_probs'].values())
n_adj     = len(rules_ex['adjacencies'])

rows = [
    ("Conjunto", "Símbolo", "Valor"),
    ("Planogramas históricos",     "|H|",         len(planograms)),
    ("Catálogo completo",          "|P|",         len(catalog)),
    ("UPCs en el surtido",         "|P_t|",       len(ex_upcs)),
    ("Charolas del template",      "|S|",         len(ex_S)),
    ("Niveles verticales",         "|L|",         7),
    ("Productos conservados",      "|P_keep|",    len(ex_P_keep)),
    ("Productos removidos",        "|P_gone|",    len(ex_P_gone)),
    ("Productos nuevos (BILP)",    "|P_new|",     len(ex_P_new)),
    ("─" * 25,                     "─"*10,        "─"*10),
    ("Variables de decisión",      "|x_{ps}|",   len(ex_P_new) * len(ex_S)),
    ("─" * 25,                     "─"*10,        "─"*10),
    ("Restricciones C1 (unicidad)","#C1",         len(ex_P_new)),
    ("Restricciones C2 (ancho)",   "#C2",         len(ex_S)),
    ("Restricciones C3 (altura)",  "#C3",         n_height_cons),
    ("Total restricciones",        "Total",       len(ex_P_new) + len(ex_S) + n_height_cons),
    ("─" * 25,                     "─"*10,        "─"*10),
    ("Parámetros π_{pℓ}",         "#π",          n_rules),
    ("Pares de adyacencia",        "|Adj|",       n_adj),
]

print(f"\n{'─'*55}")
print(f"  TAMAÑO DEL PROBLEMA (Ejemplo 1: BCO/CF/τ=4.0, 20% swap)")
print(f"{'─'*55}")
print(f"  {'Descripción':<30} {'Símbolo':^12} {'Valor':>8}")
print(f"  {'─'*50}")
for r_name, sym, val in rows[1:]:
    if str(r_name).startswith("─"):
        print(f"  {'─'*50}")
    else:
        print(f"  {r_name:<30} {sym:^12} {str(val):>8}")
print(f"{'─'*55}")


───────────────────────────────────────────────────────
  TAMAÑO DEL PROBLEMA (Ejemplo 1: BCO/CF/τ=4.0, 20% swap)
───────────────────────────────────────────────────────
  Descripción                      Símbolo       Valor
  ──────────────────────────────────────────────────
  Planogramas históricos             |H|          5614
  Catálogo completo                  |P|           470
  UPCs en el surtido                |P_t|          131
  Charolas del template              |S|            24
  Niveles verticales                 |L|             7
  Productos conservados            |P_keep|        105
  Productos removidos              |P_gone|         26
  Productos nuevos (BILP)          |P_new|          26
  ──────────────────────────────────────────────────
  Variables de decisión            |x_{ps}|        624
  ──────────────────────────────────────────────────
  Restricciones C1 (unicidad)        #C1            26
  Restricciones C2 (ancho)           #C2            24
  Restricc

## 8. Resultados Estadísticos

Evaluamos el modelo sobre los **43 planogramas originales** con 4 niveles de perturbación (10%, 20%, 30%, 40% de productos intercambiados). Para cada caso medimos:

| Métrica | Descripción |
|---|---|
| $\Phi_{\text{nivel}}$ | Adherencia histórica de nivel (promedio de $\pi_{p,\ell_s}$ para productos colocados) |
| Mode accuracy | Fracción de productos en su nivel históricamente más frecuente |
| Adjacency HR | Fracción de pares co-localizados con historial $> 0$ |
| Width feasibility | Fracción de charolas que cumplen la restricción de ancho |
| Cobertura | Productos colocados / productos en $P_t$ |

In [9]:
log("=" * 70)
log("EVALUACIÓN ESTADÍSTICA (43 planogramas × 4 niveles de perturbación)")
log("=" * 70)

swap_fracs = [0.10, 0.20, 0.30, 0.40]
results    = []

for pi, original in enumerate(planograms_orig):
    orig_upcs_i = {p.upc for s in original.shelves.values() for p in s.products}

    # Métricas del planograma original (0% swap) como línea base
    seg_plans_i = [p for p in planograms
                   if p.mueble_id == original.mueble_id
                   and p.segmento_id == original.segmento_id]
    if len(seg_plans_i) < 20:
        seg_plans_i = [p for p in planograms if p.mueble_id == original.mueble_id]

    orig_rules   = mine_placement_rules(seg_plans_i, catalog)
    orig_metrics = evaluate_rule_adherence(original, orig_rules, catalog)

    results.append({
        'plano'    : pi, 'swap': 0.0, 'source': 'original',
        'seg'      : original.segmento_id, 'mueble': original.mueble_id,
        'level'    : orig_metrics['level_rule_score'],
        'mode'     : orig_metrics['level_mode_accuracy'],
        'adj'      : orig_metrics['adjacency_hit_rate'],
        'adj_w'    : orig_metrics['adjacency_weighted'],
        'wfeas'    : orig_metrics['width_feasibility'],
        'placed'   : orig_metrics['n_products_placed'],
        'n_target' : len(orig_upcs_i),
        'coverage' : 1.0,
        'time'     : 0, 'status': 'original',
        'n_vars'   : 0, 'n_cons': 0,
    })

    for sf in swap_fracs:
        random.seed(42 + pi * 100 + int(sf * 100))
        n_swap_i = max(1, int(len(sorted(orig_upcs_i)) * sf))
        to_rm    = set(random.sample(sorted(orig_upcs_i), n_swap_i))
        avail    = sorted(all_pool - orig_upcs_i)
        repls    = set(random.sample(avail, min(n_swap_i, len(avail))))
        target_i = (orig_upcs_i - to_rm) | repls

        r = generate_planogram(
            target_upcs=target_i,
            target_tamano=original.tamano,
            target_mueble=original.mueble_id,
            target_segmento=original.segmento_id,
            historic_planograms=planograms,
            catalog=catalog,
        )

        if r['planogram'] is None:
            continue

        m = evaluate_rule_adherence(r['planogram'], r.get('rules', orig_rules), catalog)

        results.append({
            'plano'    : pi, 'swap': sf, 'source': f'gen_{sf:.0%}_swap',
            'seg'      : original.segmento_id, 'mueble': original.mueble_id,
            'level'    : m['level_rule_score'],
            'mode'     : m['level_mode_accuracy'],
            'adj'      : m['adjacency_hit_rate'],
            'adj_w'    : m['adjacency_weighted'],
            'wfeas'    : m['width_feasibility'],
            'placed'   : m['n_products_placed'],
            'n_target' : len(target_i),
            'coverage' : m['n_products_placed'] / len(target_i),
            'time'     : r['time_bilp'],
            'status'   : r['status'],
            'n_vars'   : r.get('n_vars', 0),
            'n_cons'   : r.get('n_cons', 0),
        })

    if (pi + 1) % 10 == 0:
        log(f"  {pi+1}/{len(planograms_orig)} planogramas completados")

log(f"  {len(planograms_orig)} planogramas completados — {len(results)} observaciones totales")
df = pd.DataFrame(results)

[18:20:13] ======================================================================
[18:20:13] EVALUACIÓN ESTADÍSTICA (43 planogramas × 4 niveles de perturbación)
[18:20:13] ======================================================================
[18:20:19]   10/43 planogramas completados
[18:20:25]   20/43 planogramas completados
[18:20:32]   30/43 planogramas completados
[18:20:38]   40/43 planogramas completados
[18:20:39]   43 planogramas completados — 215 observaciones totales


### 8.1 Tabla 1: Resultados por Nivel de Perturbación

In [10]:
orig_means = df[df['source'] == 'original']

# Tabla resumen con pandas
tabla1_rows = []
for src in ['original'] + [f'gen_{sf:.0%}_swap' for sf in swap_fracs]:
    sub = df[df['source'] == src]
    if len(sub) == 0:
        continue
    tabla1_rows.append({
        'Fuente'       : src,
        'Φ_nivel'      : sub['level'].mean(),
        'Mode %'       : sub['mode'].mean(),
        'Adj HR'       : sub['adj'].mean(),
        'Adj W'        : sub['adj_w'].mean(),
        'W Feasibility': sub['wfeas'].mean(),
        'Cobertura'    : sub['coverage'].mean(),
        't BILP (s)'   : sub['time'].mean(),
        'n'            : len(sub),
    })

tabla1 = pd.DataFrame(tabla1_rows).set_index('Fuente')
tabla1_fmt = tabla1.copy()
for col in ['Φ_nivel', 'Adj HR', 'Adj W', 'W Feasibility', 't BILP (s)']:
    tabla1_fmt[col] = tabla1[col].map('{:.4f}'.format)
for col in ['Mode %', 'Cobertura']:
    tabla1_fmt[col] = tabla1[col].map('{:.1%}'.format)
tabla1_fmt['n'] = tabla1['n'].astype(int)

print("\nTABLA 1: RESULTADOS POR NIVEL DE PERTURBACIÓN")
print("=" * 90)
print(tabla1_fmt.to_string())


TABLA 1: RESULTADOS POR NIVEL DE PERTURBACIÓN
             Φ_nivel Mode %  Adj HR   Adj W W Feasibility Cobertura t BILP (s)   n
Fuente                                                                            
original      0.4953  75.4%  1.0000  0.0853        1.0000    100.0%     0.0000  43
gen_10%_swap  0.4916  74.6%  0.8756  0.0723        1.0000     99.2%     0.0314  43
gen_20%_swap  0.4941  75.0%  0.7532  0.0568        1.0000     99.5%     0.0746  43
gen_30%_swap  0.5016  77.2%  0.6466  0.0449        1.0000     99.5%     0.0977  43
gen_40%_swap  0.5099  78.8%  0.5453  0.0344        1.0000     99.7%     0.1317  43


### 8.2 Tabla 2: Degradación vs. Planograma Original

In [11]:
tabla2_rows = []
for sf in swap_fracs:
    gen = df[df['source'] == f'gen_{sf:.0%}_swap']
    tabla2_rows.append({
        'Swap %'     : f'{sf:.0%}',
        'Δ Φ_nivel'  : gen['level'].mean() - orig_means['level'].mean(),
        'Δ Mode %'   : gen['mode'].mean()  - orig_means['mode'].mean(),
        'Δ Adj HR'   : gen['adj'].mean()   - orig_means['adj'].mean(),
        'Δ Cobertura': gen['coverage'].mean() - 1.0,
    })

tabla2 = pd.DataFrame(tabla2_rows).set_index('Swap %')
tabla2_fmt = tabla2.copy()
for col in ['Δ Φ_nivel', 'Δ Adj HR']:
    tabla2_fmt[col] = tabla2[col].map('{:+.4f}'.format)
for col in ['Δ Mode %', 'Δ Cobertura']:
    tabla2_fmt[col] = tabla2[col].map('{:+.1%}'.format)

print("\nTABLA 2: DEGRADACIÓN vs ORIGINAL (Δ)")
print("=" * 60)
print(tabla2_fmt.to_string())


TABLA 2: DEGRADACIÓN vs ORIGINAL (Δ)
       Δ Φ_nivel Δ Mode % Δ Adj HR Δ Cobertura
Swap %                                        
10%      -0.0037    -0.9%  -0.1244       -0.8%
20%      -0.0012    -0.4%  -0.2468       -0.5%
30%      +0.0062    +1.7%  -0.3534       -0.5%
40%      +0.0145    +3.3%  -0.4547       -0.3%


### 8.3 Tabla 3: Resultados por Segmento (20% swap)

In [12]:
gen20 = df[df['swap'] == 0.20]
if len(gen20) > 0:
    seg_summary = gen20.groupby('seg').agg({
        'level'   : 'mean',
        'mode'    : 'mean',
        'adj'     : 'mean',
        'coverage': 'mean',
        'time'    : 'mean',
        'n_vars'  : 'mean',
        'n_cons'  : 'mean',
    }).round(4)

    seg_fmt = seg_summary.copy()
    for col in ['level', 'adj', 'time', 'n_vars', 'n_cons']:
        seg_fmt[col] = seg_summary[col].map('{:.4f}'.format)
    seg_fmt['mode']     = seg_summary['mode'].map('{:.1%}'.format)
    seg_fmt['coverage'] = seg_summary['coverage'].map('{:.1%}'.format)
    seg_fmt = seg_fmt.rename(columns={
        'level': 'Φ_nivel', 'mode': 'Mode%', 'adj': 'Adj HR',
        'coverage': 'Cobertura', 'time': 't(s)', 'n_vars': '|vars|', 'n_cons': '|cons|'
    })

    print("\nTABLA 3: RESULTADOS POR SEGMENTO (20% swap)")
    print("=" * 70)
    print(seg_fmt.to_string())

# Guardar CSV completo
df.to_csv(os.path.join(OUTPUT_DIR, "full_results.csv"), index=False)
log(f"CSV guardado: {OUTPUT_DIR}/full_results.csv")


TABLA 3: RESULTADOS POR SEGMENTO (20% swap)
    Φ_nivel  Mode%  Adj HR Cobertura    t(s)    |vars|    |cons|
seg                                                             
BCO  0.4895  73.6%  0.7516     98.9%  0.0795  554.0000  183.2000
CLA  0.4911  74.0%  0.7585     99.4%  0.0792  533.5714  167.0714
HRN  0.4988  76.5%  0.7500     99.9%  0.0686  588.1053  164.2632
[18:20:39] CSV guardado: test_output/final_model/full_results.csv


## 9. Análisis de Sensibilidad

### 9.1 Por qué el BILP no tiene dualidad LP clásica

En un programa entero binario (BILP), las variables $x_{ps} \in \{0, 1\}$ son discretas. La relajación LP (donde $0 \leq x_{ps} \leq 1$) sí produce precios sombra, pero estos no son directamente interpretables para el problema entero: el **integrality gap** puede hacer que los duales de la relajación sean engañosos.

**Alternativa propuesta:** Realizamos dos tipos de análisis paramétrico:

1. **Sensibilidad a $\lambda_1$ y $\lambda_2$** — ¿Cómo cambia la cobertura y la adherencia al variar el peso de cada componente de la FO?
2. **Sensibilidad al ancho residual** — ¿Cuántos productos se pierden si se reduce el espacio disponible en los anaqueles?

### 9.2 Análisis Paramétrico sobre $\lambda_1$ y $\lambda_2$

Fijamos el Ejemplo 1 (BCO/CF/τ=4.0, 20% swap) y variamos $\lambda_1 \in [0, 3]$ con $\lambda_2 = 1$, luego $\lambda_2 \in [0, 3]$ con $\lambda_1 = 1$.

In [13]:
log("Análisis de sensibilidad sobre λ1 y λ2...")

lambda_vals = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]
sens_rows   = []

# Variar λ1 con λ2=1 fijo
for lam1 in lambda_vals:
    r_s = generate_planogram(
        target_upcs=target1, target_tamano=ex1.tamano,
        target_mueble=ex1.mueble_id, target_segmento=ex1.segmento_id,
        historic_planograms=planograms, catalog=catalog,
        lambda1=lam1, lambda2=1.0,
    )
    m_s = evaluate_rule_adherence(r_s['planogram'], r_s['rules'], catalog)
    sens_rows.append({
        'Experimento': f'λ1={lam1:.1f}, λ2=1.0',
        'λ1': lam1, 'λ2': 1.0,
        'Colocados'  : r_s['n_placed'],
        'Φ_nivel'    : m_s['level_rule_score'],
        'Mode %'     : m_s['level_mode_accuracy'],
        'Adj HR'     : m_s['adjacency_hit_rate'],
        'Cobertura'  : m_s['n_products_placed'] / len(target1),
    })

# Variar λ2 con λ1=1 fijo
for lam2 in lambda_vals:
    r_s = generate_planogram(
        target_upcs=target1, target_tamano=ex1.tamano,
        target_mueble=ex1.mueble_id, target_segmento=ex1.segmento_id,
        historic_planograms=planograms, catalog=catalog,
        lambda1=1.0, lambda2=lam2,
    )
    m_s = evaluate_rule_adherence(r_s['planogram'], r_s['rules'], catalog)
    sens_rows.append({
        'Experimento': f'λ1=1.0, λ2={lam2:.1f}',
        'λ1': 1.0, 'λ2': lam2,
        'Colocados'  : r_s['n_placed'],
        'Φ_nivel'    : m_s['level_rule_score'],
        'Mode %'     : m_s['level_mode_accuracy'],
        'Adj HR'     : m_s['adjacency_hit_rate'],
        'Cobertura'  : m_s['n_products_placed'] / len(target1),
    })

df_sens = pd.DataFrame(sens_rows)
df_sens_fmt = df_sens.copy()
df_sens_fmt['Φ_nivel'] = df_sens['Φ_nivel'].map('{:.4f}'.format)
df_sens_fmt['Adj HR']  = df_sens['Adj HR'].map('{:.4f}'.format)
df_sens_fmt['Mode %']  = df_sens['Mode %'].map('{:.1%}'.format)
df_sens_fmt['Cobertura'] = df_sens['Cobertura'].map('{:.1%}'.format)

print("\nANÁLISIS DE SENSIBILIDAD — Variación de λ1 y λ2")
print("=" * 80)
print(df_sens_fmt[['Experimento', 'Colocados', 'Φ_nivel', 'Mode %', 'Adj HR', 'Cobertura']].to_string(index=False))

[18:20:39] Análisis de sensibilidad sobre λ1 y λ2...

ANÁLISIS DE SENSIBILIDAD — Variación de λ1 y λ2
   Experimento  Colocados Φ_nivel Mode % Adj HR Cobertura
λ1=0.0, λ2=1.0        129  0.4650  67.5% 0.7377     98.5%
λ1=0.5, λ2=1.0        129  0.5023  74.8% 0.7548     98.5%
λ1=1.0, λ2=1.0        129  0.5036  74.8% 0.7621     98.5%
λ1=1.5, λ2=1.0        129  0.5053  75.6% 0.7492     98.5%
λ1=2.0, λ2=1.0        129  0.5053  75.6% 0.7588     98.5%
λ1=3.0, λ2=1.0        129  0.5053  75.6% 0.7588     98.5%
λ1=1.0, λ2=0.0        129  0.5053  75.6% 0.7249     98.5%
λ1=1.0, λ2=0.5        129  0.5053  75.6% 0.7492     98.5%
λ1=1.0, λ2=1.0        129  0.5036  74.8% 0.7621     98.5%
λ1=1.0, λ2=1.5        129  0.5023  74.8% 0.7548     98.5%
λ1=1.0, λ2=2.0        129  0.5023  74.8% 0.7548     98.5%
λ1=1.0, λ2=3.0        129  0.5023  74.8% 0.7548     98.5%


### 9.3 Sensibilidad al Ancho Residual $W_s^{\text{rem}}$

In [14]:
log("Análisis de sensibilidad al ancho residual...")

# Simulamos reducción del ancho disponible modificando artificialmente el catálogo
# (equivale a aumentar el ancho de los productos P_keep, reduciendo W_s^rem)
ancho_fracs = [1.0, 0.90, 0.80, 0.70, 0.60]  # fracción de ancho residual disponible
ancho_rows  = []

for frac in ancho_fracs:
    # Crear un catálogo modificado donde los productos P_keep son más anchos
    # Esto reduce efectivamente el W_s^rem disponible para P_new
    cat_mod = {}
    for upc, info in catalog.items():
        # Escalar ancho para P_keep equivale a reducir W^rem en (1-frac)
        if upc in (target1 & ex_P_T):
            cat_mod[upc] = ProductInfo(
                upc=info.upc,
                description=info.description,
                width=info.width / frac,  # ancho aumentado → menos espacio residual
                height=info.height,
            )
        else:
            cat_mod[upc] = info

    r_a = generate_planogram(
        target_upcs=target1, target_tamano=ex1.tamano,
        target_mueble=ex1.mueble_id, target_segmento=ex1.segmento_id,
        historic_planograms=planograms, catalog=cat_mod,
        lambda1=1.0, lambda2=1.0,
    )
    if r_a['planogram'] is None:
        ancho_rows.append({'W_rem (%)': f'{frac:.0%}', 'Colocados': 'Infeasible',
                           'P_new placed': '-', 'Cobertura': '-'})
    else:
        ancho_rows.append({
            'W_rem (%)': f'{frac:.0%}',
            'Colocados': r_a['n_placed'],
            'P_new placed': r_a['n_new_placed'],
            'Cobertura': f"{(r_a['n_placed'] / len(target1)):.1%}",
        })

df_ancho = pd.DataFrame(ancho_rows)
print("\nSENSIBILIDAD AL ANCHO RESIDUAL (Ejemplo 1, 20% swap)")
print("=" * 55)
print(df_ancho.to_string(index=False))
print("\nNota: W_rem 100% = capacidad nominal; valores menores simulan")
print("anaqueles más llenos o productos P_keep con más frentes.")

[18:20:41] Análisis de sensibilidad al ancho residual...

SENSIBILIDAD AL ANCHO RESIDUAL (Ejemplo 1, 20% swap)
W_rem (%)  Colocados  P_new placed Cobertura
     100%        129            24     98.5%
      90%        131            26    100.0%
      80%        128            23     97.7%
      70%        117            12     89.3%
      60%        131            26    100.0%

Nota: W_rem 100% = capacidad nominal; valores menores simulan
anaqueles más llenos o productos P_keep con más frentes.


## 10. Valor Esperado de la Ubicación del Producto

El modelo TA-BILP incorpora incertidumbre a través de las probabilidades históricas $\pi_{p\ell}$. En esta sección calculamos explícitamente el **valor esperado del nivel** de cada producto y analizamos cuánto se desvía el planograma generado de ese óptimo estocástico.

### 10.1 Definición

Para cada producto $p$ con al menos una observación histórica, definimos:

$$\mathbb{E}[\ell_p] = \sum_{\ell \in L} \ell \cdot \pi_{p\ell}$$

Este valor representa el nivel "esperado" donde históricamente se coloca el producto. Un planograma ideal colocaría cada producto en $\ell = \text{round}(\mathbb{E}[\ell_p])$.

### 10.2 Incertidumbre asociada

La varianza del nivel:

$$\text{Var}[\ell_p] = \sum_{\ell \in L} \ell^2 \cdot \pi_{p\ell} - \mathbb{E}[\ell_p]^2$$

Un producto con $\text{Var}[\ell_p] \approx 0$ siempre aparece en el mismo nivel; uno con varianza alta tiene ubicación incierta.

### 10.3 Cálculo

**Añadido respecto al script original**: esta sección computa y visualiza la distribución de $\mathbb{E}[\ell_p]$ y $\text{Var}[\ell_p]$ para todos los productos del catálogo.

In [15]:
# Calcular reglas de nivel para el mueble CF (el más representativo)
rules_cf = mine_placement_rules(
    [p for p in planograms if p.mueble_id == 'CF'], catalog
)
lp_cf = rules_cf['level_probs']  # {upc: {level: prob}}

ev_rows = []
for upc, level_dist in lp_cf.items():
    if upc not in catalog:
        continue
    levels = list(level_dist.keys())
    probs  = list(level_dist.values())

    # Valor esperado E[ℓ_p]
    e_level = sum(l * p for l, p in zip(levels, probs))

    # Varianza Var[ℓ_p]
    e_level2 = sum(l**2 * p for l, p in zip(levels, probs))
    var_level = e_level2 - e_level**2

    # Nivel modal (el más frecuente)
    mode_level = max(level_dist, key=level_dist.get)
    prob_mode  = level_dist[mode_level]

    ev_rows.append({
        'UPC'        : upc,
        'width'      : catalog[upc].width,
        'height'     : catalog[upc].height,
        'E[nivel]'   : e_level,
        'Var[nivel]' : var_level,
        'Nivel_moda' : mode_level,
        'P(moda)'    : prob_mode,
        'n_niveles'  : len(levels),
    })

df_ev = pd.DataFrame(ev_rows)
print(f"Productos con reglas históricas (CF): {len(df_ev)}")
print(f"\nEstadísticos de E[nivel]:")
print(df_ev['E[nivel]'].describe().round(3))
print(f"\nEstadísticos de Var[nivel]:")
print(df_ev['Var[nivel]'].describe().round(3))

Productos con reglas históricas (CF): 456

Estadísticos de E[nivel]:
count    456.000
mean       2.420
std        1.104
min        0.000
25%        1.293
50%        2.538
75%        3.316
max        6.000
Name: E[nivel], dtype: float64

Estadísticos de Var[nivel]:
count    456.000
mean       1.733
std        0.984
min        0.000
25%        1.084
50%        1.417
75%        2.501
max        7.688
Name: Var[nivel], dtype: float64


In [16]:
# Productos con ubicación más incierta (alta varianza)
print("\nTop 10 productos con MAYOR incertidumbre de nivel (Var[ℓ] alta):")
top_var = df_ev.nlargest(10, 'Var[nivel]')[
    ['UPC', 'height', 'width', 'E[nivel]', 'Var[nivel]', 'P(moda)']
].reset_index(drop=True)
print(top_var.round(3).to_string(index=False))

print("\nTop 10 productos con MENOR incertidumbre (posición casi fija):")
bot_var = df_ev.nsmallest(10, 'Var[nivel]')[
    ['UPC', 'height', 'width', 'E[nivel]', 'Var[nivel]', 'P(moda)']
].reset_index(drop=True)
print(bot_var.round(3).to_string(index=False))


Top 10 productos con MAYOR incertidumbre de nivel (Var[ℓ] alta):
          UPC  height  width  E[nivel]  Var[nivel]  P(moda)
7501055300921    32.0    9.0     3.250       7.688    0.500
7501055331901    34.3   10.4     2.000       6.000    0.500
 792313001002    37.0   12.2     1.667       5.556    0.667
7501055314157    35.5   10.3     2.571       5.388    0.429
7501055335176    16.0    6.7     3.000       4.000    0.500
7501055353699    33.0    9.0     3.000       4.000    0.500
7501055349906    11.0    5.0     2.000       4.000    0.500
7501055314805    33.0    9.5     2.182       3.967    0.545
7501055358984    15.6    6.0     3.715       3.679    0.611
7501055361540    15.8    5.8     3.675       3.601    0.587

Top 10 productos con MENOR incertidumbre (posición casi fija):
          UPC  height  width  E[nivel]  Var[nivel]  P(moda)
7501055382163    34.4    9.6       1.0         0.0      1.0
7501055311132    15.6    6.0       5.0         0.0      1.0
7501055302659    15.6    6.0  

In [17]:
# Calcular desviación entre nivel real en planogramas generados y E[nivel]
gen_df_eval = df[df['source'] != 'original'].copy()

# Para cada planograma generado en la evaluación estadística (20% swap),
# calcular cuánto difiere el nivel asignado de E[nivel] para los productos
all_deviations = []
for pi, original in enumerate(planograms_orig):
    random.seed(42 + pi * 100 + int(0.20 * 100))
    orig_upcs_i = {p.upc for s in original.shelves.values() for p in s.products}
    n_swap_i = max(1, int(len(orig_upcs_i) * 0.20))
    to_rm = set(random.sample(sorted(orig_upcs_i), n_swap_i))
    avail = sorted(all_pool - orig_upcs_i)
    repls = set(random.sample(avail, min(n_swap_i, len(avail))))
    target_i = (orig_upcs_i - to_rm) | repls

    r = generate_planogram(
        target_upcs=target_i, target_tamano=original.tamano,
        target_mueble=original.mueble_id, target_segmento=original.segmento_id,
        historic_planograms=planograms, catalog=catalog,
    )
    if r['planogram'] is None:
        continue

    for shelf in r['planogram'].shelves.values():
        for prod in shelf.products:
            if prod.upc in lp_cf:
                e_lv = sum(l * p for l, p in lp_cf[prod.upc].items())
                all_deviations.append(abs(shelf.level - e_lv))

dev_arr = np.array(all_deviations)
print(f"\nDESVIACIÓN |nivel_asignado - E[nivel]| (evaluación 20% swap, mueble CF)")
print(f"  n productos evaluados : {len(dev_arr)}")
print(f"  Media                 : {dev_arr.mean():.3f} niveles")
print(f"  Mediana               : {np.median(dev_arr):.3f} niveles")
print(f"  P90                   : {np.percentile(dev_arr, 90):.3f} niveles")
print(f"  Productos en E[nivel] ± 0.5: {(dev_arr <= 0.5).mean():.1%}")


DESVIACIÓN |nivel_asignado - E[nivel]| (evaluación 20% swap, mueble CF)
  n productos evaluados : 5158
  Media                 : 0.703 niveles
  Mediana               : 0.609 niveles
  P90                   : 1.356 niveles
  Productos en E[nivel] ± 0.5: 45.4%


## 11. Visualizaciones

Se presentan cuatro gráficas que resumen el desempeño del modelo:
1. Distribución de tiempos de solución del BILP.
2. Cobertura vs. nivel de perturbación.
3. $\Phi_{\text{nivel}}$ vs. nivel de perturbación.
4. Distribución del valor esperado $\mathbb{E}[\ell_p]$ y varianza $\text{Var}[\ell_p]$.

In [18]:
gen_df_plot = df[df['source'] != 'original']
orig_mean_level = orig_means['level'].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("TA-BILP — Resumen de Resultados", fontsize=14, fontweight='bold')

# ── Gráfica 1: Distribución de tiempos BILP ────────────────────────────
ax = axes[0, 0]
ax.hist(gen_df_plot['time'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(gen_df_plot['time'].mean(), color='crimson', ls='--', lw=1.5,
           label=f'media = {gen_df_plot["time"].mean():.3f}s')
ax.set_xlabel('Tiempo BILP (s)')
ax.set_ylabel('Frecuencia')
ax.set_title(f'Distribución de tiempo de solución (n={len(gen_df_plot)})')
ax.legend()

# ── Gráfica 2: Cobertura vs. perturbación ──────────────────────────────
ax = axes[0, 1]
colors2 = plt.cm.viridis(np.linspace(0.2, 0.8, len(swap_fracs)))
for i, sf in enumerate(swap_fracs):
    sub = gen_df_plot[gen_df_plot['swap'] == sf]
    ax.scatter([sf]*len(sub), sub['coverage'], alpha=0.5, s=25, color=colors2[i],
               label=f'{sf:.0%}')
ax.axhline(1.0, color='gray', ls='--', lw=1, alpha=0.5, label='Cobertura ideal')
ax.set_xlabel('Fracción de productos intercambiados (swap %)')
ax.set_ylabel('Cobertura')
ax.set_title('Cobertura vs. Perturbación')
ax.set_ylim(0.85, 1.02)
ax.legend(title='Swap', fontsize=8)

# ── Gráfica 3: Φ_nivel vs. perturbación ────────────────────────────────
ax = axes[1, 0]
for i, sf in enumerate(swap_fracs):
    sub = gen_df_plot[gen_df_plot['swap'] == sf]
    ax.scatter([sf]*len(sub), sub['level'], alpha=0.5, s=25, color=colors2[i])
ax.axhline(orig_mean_level, color='crimson', ls='--', lw=1.5,
           label=f'original = {orig_mean_level:.3f}')
ax.set_xlabel('Fracción de productos intercambiados (swap %)')
ax.set_ylabel('$\Phi_{nivel}$')
ax.set_title('Adherencia histórica $\Phi_{nivel}$ vs. Perturbación')
ax.legend()

# ── Gráfica 4: E[nivel] y Var[nivel] ───────────────────────────────────
ax = axes[1, 1]
scatter = ax.scatter(df_ev['E[nivel]'], df_ev['Var[nivel]'],
                     c=df_ev['height'], cmap='plasma', alpha=0.6, s=20)
cb = fig.colorbar(scatter, ax=ax)
cb.set_label('Altura del producto (cm)')
ax.set_xlabel('$\mathbb{E}[\ell_p]$ — Nivel esperado')
ax.set_ylabel('$\mathrm{Var}[\ell_p]$ — Varianza del nivel')
ax.set_title('Valor esperado vs. Incertidumbre de nivel (por producto)')

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "stats_overview.png"), dpi=150, bbox_inches='tight')
plt.close(fig)
log(f"Gráfica guardada: {OUTPUT_DIR}/stats_overview.png")

[18:20:48] Gráfica guardada: test_output/final_model/stats_overview.png


In [19]:
# ── Gráfica adicional: distribución de E[nivel] por altura ─────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4))
fig2.suptitle("Distribución del Valor Esperado de Ubicación", fontsize=13)

# Histograma de E[nivel]
axes2[0].hist(df_ev['E[nivel]'], bins=20, color='teal', edgecolor='white', alpha=0.8)
axes2[0].set_xlabel('$\mathbb{E}[\ell_p]$ — Nivel esperado')
axes2[0].set_ylabel('Número de productos')
axes2[0].set_title('Distribución de E[nivel] en el catálogo (CF)')

# E[nivel] vs altura del producto
axes2[1].scatter(df_ev['height'], df_ev['E[nivel]'],
                  c=df_ev['Var[nivel]'], cmap='Reds', alpha=0.6, s=25)
axes2[1].set_xlabel('Altura del producto (cm)')
axes2[1].set_ylabel('$\mathbb{E}[\ell_p]$')
axes2[1].set_title('Correlación altura → nivel esperado (confirma r≈-0.90)')

fig2.tight_layout()
fig2.savefig(os.path.join(OUTPUT_DIR, "expected_level.png"), dpi=150, bbox_inches='tight')
plt.close(fig2)
log(f"Gráfica guardada: {OUTPUT_DIR}/expected_level.png")

# Correlación empírica altura vs E[nivel]
valid = df_ev.dropna(subset=['height', 'E[nivel]'])
corr = valid[['height', 'E[nivel]']].corr().iloc[0, 1]
print(f"\nCorrelación empírica altura vs E[nivel]: r = {corr:.3f}")
print("(El modelo asume r = -0.90; correlación negativa confirma el supuesto)")

[18:20:48] Gráfica guardada: test_output/final_model/expected_level.png

Correlación empírica altura vs E[nivel]: r = -0.842
(El modelo asume r = -0.90; correlación negativa confirma el supuesto)


## 12. Supuestos y Conclusiones

### 12.1 Supuestos del Modelo

La tabla siguiente distingue entre supuestos **explícitos** (enunciados y verificables) e **implícitos** (asumidos sin validación directa).

In [20]:
print('''
SUPUESTOS DEL MODELO TA-BILP
============================

 #  Supuesto                                                              Tipo
 ----------------------------------------------------------------- ----------
 1  Todo producto tiene dimensiones conocidas (w_p, h_p)          Explicito
 2  Estructura del mueble fija: tau, |S|, (W_s, H_s)             Explicito
 3  Frentes f_p heredados del template (f_p = 1 para P_new)      Explicito
 4  CF y CFC no comparten reglas ni templates                     Explicito
 5  Segmentos con <20 planos usan fallback al nivel de mueble     Explicito
 6  Template T* se selecciona por maximo overlap (P1)             Explicito
 7  P_keep se fija en su charola original (no se re-optimiza)     Explicito
 8  Dimensiones del catalogo son estaticas y confiables           Implicito
 9  Un UPC aparece maximo una vez por charola                     Explicito
10  La correlacion alto-nivel (r=-0.90) es estable entre seg.     Implicito
''')


SUPUESTOS DEL MODELO TA-BILP

 #  Supuesto                                                              Tipo
 ----------------------------------------------------------------- ----------
 1  Todo producto tiene dimensiones conocidas (w_p, h_p)          Explicito
 2  Estructura del mueble fija: tau, |S|, (W_s, H_s)             Explicito
 3  Frentes f_p heredados del template (f_p = 1 para P_new)      Explicito
 4  CF y CFC no comparten reglas ni templates                     Explicito
 5  Segmentos con <20 planos usan fallback al nivel de mueble     Explicito
 6  Template T* se selecciona por maximo overlap (P1)             Explicito
 7  P_keep se fija en su charola original (no se re-optimiza)     Explicito
 8  Dimensiones del catalogo son estaticas y confiables           Implicito
 9  Un UPC aparece maximo una vez por charola                     Explicito
10  La correlacion alto-nivel (r=-0.90) es estable entre seg.     Implicito



### 12.2 Conclusiones

In [21]:
gen_all_eval = df[df['source'] != 'original']
gen20_eval   = df[df['swap'] == 0.20]
gen40_eval   = df[df['swap'] == 0.40]

print(f"""
CONCLUSIONES
============

1. CALIDAD DE LA SOLUCIÓN (20% swap)
   ─────────────────────────────────
   Φ_nivel promedio  : {gen20_eval['level'].mean():.3f}  (original: {orig_means['level'].mean():.3f})
   Mode accuracy     : {gen20_eval['mode'].mean():.1%}  (original: {orig_means['mode'].mean():.1%})
   Adjacency HR      : {gen20_eval['adj'].mean():.3f}  (original: {orig_means['adj'].mean():.3f})
   Cobertura         : {gen20_eval['coverage'].mean():.1%}  de productos objetivo colocados
   Width feasibility : {gen20_eval['wfeas'].mean():.3f}

2. DESEMPEÑO COMPUTACIONAL
   ───────────────────────
   Tiempo promedio BILP : {gen_all_eval['time'].mean():.3f} s
   Tiempo máximo        : {gen_all_eval['time'].max():.3f} s
   Variables promedio   : {gen_all_eval['n_vars'].mean():.0f}
   Restricciones prom.  : {gen_all_eval['n_cons'].mean():.0f}
   Todos resueltos a optimalidad (status = Optimal)

3. VENTAJAS DEL MODELO TA-BILP
   ─────────────────────────────
   • Auto-contenido: recibe solo BD + parámetros de entrada.
   • Resuelve en <1 s vs. horas para un BILP from-scratch.
   • Preserva co-ocurrencia naturalmente al anclar P_keep.
   • Óptimo garantizado (branch-and-bound, gap ≤ 1%).
   • Escala a cualquier tamaño de mueble (1.0 – 6.5 puertas).

4. DEGRADACIÓN CONTROLADA
   ───────────────────────
   Con 20% de productos intercambiados:
     Δ Φ_nivel ≈ {gen20_eval['level'].mean()-orig_means['level'].mean():+.3f}
     Δ Adj HR  ≈ {gen20_eval['adj'].mean()-orig_means['adj'].mean():+.3f}
   Con 40% de intercambio:
     Δ Φ_nivel ≈ {gen40_eval['level'].mean()-orig_means['level'].mean():+.3f}
     Δ Adj HR  ≈ {gen40_eval['adj'].mean()-orig_means['adj'].mean():+.3f}
   La degradación es proporcional al % de perturbación.

5. LIMITACIONES
   ─────────────
   • P_keep se fija sin re-optimización: no aprovecha intercambios entre niveles.
   • La función ideal_y() usa umbrales discretos, no una función continua.
   • El análisis de sensibilidad sobre λ es paramétrico, no analítico.
   • La correlación r=-0.90 asumida puede variar por segmento.

6. MEJORAS FUTURAS
   ─────────────────
   • Incluir demanda real como coeficiente en Φ_cob para priorizar SKUs de mayor rotación.
   • Optimizar también P_keep mediante una BILP de 2 etapas.
   • Incorporar restricciones de planograma mínimo/máximo por categoría.
   • Sustituir ideal_y() por una regresión calibrada por segmento.
""")


CONCLUSIONES

1. CALIDAD DE LA SOLUCIÓN (20% swap)
   ─────────────────────────────────
   Φ_nivel promedio  : 0.494  (original: 0.495)
   Mode accuracy     : 75.0%  (original: 75.4%)
   Adjacency HR      : 0.753  (original: 1.000)
   Cobertura         : 99.5%  de productos objetivo colocados
   Width feasibility : 1.000

2. DESEMPEÑO COMPUTACIONAL
   ───────────────────────
   Tiempo promedio BILP : 0.084 s
   Tiempo máximo        : 0.413 s
   Variables promedio   : 705
   Restricciones prom.  : 204
   Todos resueltos a optimalidad (status = Optimal)

3. VENTAJAS DEL MODELO TA-BILP
   ─────────────────────────────
   • Auto-contenido: recibe solo BD + parámetros de entrada.
   • Resuelve en <1 s vs. horas para un BILP from-scratch.
   • Preserva co-ocurrencia naturalmente al anclar P_keep.
   • Óptimo garantizado (branch-and-bound, gap ≤ 1%).
   • Escala a cualquier tamaño de mueble (1.0 – 6.5 puertas).

4. DEGRADACIÓN CONTROLADA
   ───────────────────────
   Con 20% de productos int